# 08. Python Dictionaries (5+ Years Interview Guide)
Deep architectural guide to hash tables, compact dictionary memory layout, collision resolution, O(1) lookups, dictionary views, and Python 3.9+ merge operators.

### Key 5-Year Interview Concepts Covered:
- **Compact Hash Table Internals**: Separate hash/index sparse table and dense key-value array (since Python 3.6), preserving insertion order.
- **Hashable Protocol (`__hash__` and `__eq__`)**: Why dictionary keys must be immutable and hashable.
- **Key Retrieval Strategies**: Direct indexing `[]` vs `.get()` vs `.setdefault()` vs `collections.defaultdict`.
- **Dictionary Merging**: `dict.update()` vs dictionary unpacking `{
**d1, **d2}` vs Python 3.9+ union operator `d1 | d2`.

This notebook uses the shared Fintech dataset `data/raw_transactions.csv` for interview scenario problems at the end.

In [1]:
# Setup: Locate the Shared Dataset
import os
csv_path = 'data/raw_transactions.csv' if os.path.exists('data/raw_transactions.csv') else '../data/raw_transactions.csv'
print("Using CSV file path:", csv_path)

Using CSV file path: data/raw_transactions.csv


### 1. Dictionary Architecture & Key-Value Mappings
**Explanation**: Python dictionaries (`dict`) are high-performance hash tables mapping unique keys to values. Since Python 3.6, CPython implements a compact dictionary layout using two arrays: a sparse indices table and a dense array storing `[hash, key, value]` in insertion order. This reduces memory usage by 20-30% and guarantees deterministic insertion order iteration.

**Syntax**: `my_dict = {key: value}` / `my_dict = dict(key=value)`

In [2]:
transaction_ledger_dict = {'TX101': 250.0}
print(transaction_ledger_dict)

{'TX101': 250.0}


### 2. Hashable Key Constraints & Collision Resolution
**Explanation**: To be used as a dictionary key, an object must implement the hash protocol: `__hash__()` and `__eq__()`. The hash must remain constant throughout the object's lifecycle (immutability). Mutable objects like lists and dicts raise `TypeError: unhashable type`. When two distinct keys produce the same hash index (hash collision), CPython resolves it using open addressing with pseudo-random perturbation probing.

**Syntax**: `hash(key_object)  # Must be integer; object must be immutable`

In [3]:
try:
    mutable_keys_dict = {[1]: 2}
except TypeError as error_message:
    print('Unhashable error caught:', error_message)

Unhashable error caught: unhashable type: 'list'


### 3. Key Lookup Access & O(1) Complexity
**Explanation**: Accessing a dictionary by key `my_dict[key]` computes `hash(key)`, maps it to an index in the sparse table, and retrieves the value pointer in average O(1) constant time. If the key does not exist, it raises a `KeyError`.

**Syntax**: `value = my_dict[key]  # O(1) average lookup`

In [4]:
transaction_ledger_dict = {'TX101': 250.0}
print(transaction_ledger_dict['TX101'])

250.0


### 4. Modifying & Adding Entries
**Explanation**: Setting `my_dict[key] = value` updates the existing key if found or inserts a new key-value pair. If the dictionary's load factor exceeds 2/3, CPython automatically resizes the table (typically doubling capacity) and re-indexes elements.

**Syntax**: `my_dict[key] = new_value`

In [5]:
transaction_ledger_dict = {}
transaction_ledger_dict['TX101'] = 10.0
print(transaction_ledger_dict)

{'TX101': 10.0}


### 5. Safe Key Lookups with `.get()`
**Explanation**: The `.get(key, default=None)` method retrieves a value without risking a `KeyError`. If the key exists, it returns the value; otherwise, it returns the specified fallback default. In production, use `.get()` whenever keys might be missing from incoming API JSON payloads.

**Syntax**: `value = my_dict.get(key, fallback_default)`

In [6]:
safe_lookup_dictionary = {'a': 1}
print(safe_lookup_dictionary.get('b', 0))

0


### 6. Initializing Default States (`.setdefault()`)
**Explanation**: `.setdefault(key, default_value)` checks if `key` exists. If present, it returns the existing value. If missing, it inserts `key: default_value` and returns `default_value`. It is commonly used for grouping data into lists: `my_dict.setdefault(category, []).append(item)`.

**Syntax**: `my_dict.setdefault(category, []).append(item)`

In [7]:
safe_lookup_dictionary = {'a': 1}
safe_lookup_dictionary.setdefault('b', 2)
print(safe_lookup_dictionary)

{'a': 1, 'b': 2}


### 7. Removing Keys (`pop()` vs `del` vs `popitem()`)
**Explanation**: `.pop(key, default)` removes the key and returns its value (or default if missing). `del my_dict[key]` deletes the key but raises `KeyError` if missing. `.popitem()` removes and returns the last inserted `(key, value)` pair in LIFO order in O(1) time.

**Syntax**: `val = my_dict.pop(key, default)` / `del my_dict[key]` / `k, v = my_dict.popitem()`

In [8]:
safe_lookup_dictionary = {'a': 1}
safe_lookup_dictionary.pop('a')
print(safe_lookup_dictionary)

{}


### 8. Clearing Dictionaries (`.clear()`)
**Explanation**: `.clear()` removes all key-value entries in-place, reducing length to 0 while keeping the same dictionary object and memory address reference. This is different from reassigning `my_dict = {}`, which creates a new object and leaves old references intact.

**Syntax**: `my_dict.clear()`

In [9]:
safe_lookup_dictionary = {'a': 1}
safe_lookup_dictionary.clear()
print(safe_lookup_dictionary)

{}


### 9. Iterating Keys, Values, and Items
**Explanation**: Dictionaries provide methods returning dynamic view objects: `.keys()` (all keys), `.values()` (all values), and `.items()` (`(key, value)` pairs). Iterating directly over `for k in my_dict:` iterates over keys with minimal overhead.

**Syntax**: `for k, v in my_dict.items(): ...`

In [10]:
safe_lookup_dictionary = {'a': 1}
print(list(safe_lookup_dictionary.items()))

[('a', 1)]


### 10. Dictionary Comprehensions
**Explanation**: Dictionary comprehensions `{key_expr: value_expr for item in iterable if cond}` construct dictionaries declaratively in a single step at C speed. They are widely used in data pipelines for indexing lists of records by unique ID.

**Syntax**: `{row['id']: row['amount'] for row in transactions if row['valid']}`

In [11]:
print({x: x**2 for x in range(3)})

{0: 0, 1: 1, 2: 4}


### 11. Merging Dictionaries (`.update()`)
**Explanation**: `.update(other_dict)` modifies the dictionary in-place by overwriting existing keys and inserting new keys from the source dictionary or keyword arguments.

**Syntax**: `my_dict.update(other_dict)` / `my_dict.update(k1=v1, k2=v2)`

In [12]:
dict_one = {'a': 1}
dict_one.update({'b': 2})
print(dict_one)

{'a': 1, 'b': 2}


### 12. Dictionary Merge & Update Operators (Python 3.9+ `|` and `|=`)
**Explanation**: PEP 584 introduced the merge operator `|` (returns a new combined dictionary where right-hand values overwrite left-hand values) and update operator `|=` (modifies left operand in-place). This is cleaner than `{**d1, **d2}` and supports mixed dictionary types.

**Syntax**: `merged_dict = d1 | d2` / `d1 |= d2  # In-place merge`

In [13]:
dict_one = {'a': 1}
dict_two = {'b': 2}
print(dict_one | dict_two)

{'a': 1, 'b': 2}


### 13. Shallow Copy of Dictionaries (`.copy()`)
**Explanation**: `.copy()` or `dict(my_dict)` creates a new top-level dictionary container with copied key/value pointers. Modifying keys in the copy does not affect the original, but modifying nested mutable values (like nested dicts or lists) mutates the shared underlying objects.

**Syntax**: `dict_copy = my_dict.copy()`

In [14]:
dictionary_variable = {'a': []}
copied_dict = dictionary_variable.copy()
copied_dict['a'].append(1)
print('c:', copied_dict, 'd:', dictionary_variable)

c: {'a': [1]} d: {'a': [1]}


### 14. Key Membership Validation (`in`)
**Explanation**: Checking `key in my_dict` executes a hash lookup in O(1) average time. Never do `key in my_dict.keys()`, as `in my_dict` directly checks keys without redundant method lookups.

**Syntax**: `if 'target_key' in my_dict: ...  # O(1) hash check`

In [15]:
dictionary_variable = {'a': 1}
print('a' in dictionary_variable)

True


### 15. Dictionary View Objects & Set Operations
**Explanation**: The objects returned by `dict.keys()`, `dict.values()`, and `dict.items()` are dynamic views that reflect changes made to the dictionary immediately without copying. Furthermore, `dict.keys()` and `dict.items()` act like sets, supporting set operations: `d1.keys() & d2.keys()` finds shared keys in O(min(len(d1), len(d2))) time.

**Syntax**: `shared_keys = d1.keys() & d2.keys()` / `diff_keys = d1.keys() - d2.keys()`

In [16]:
dictionary_variable = {'a': 1}
keys_view = dictionary_variable.keys()
dictionary_variable['b'] = 2
print('Dynamic updates:', keys_view)

Dynamic updates: dict_keys(['a', 'b'])


## Section 3: Fintech Senior Interview Scenarios
**Explanation**: High-performance dictionary aggregation, frequency mapping with `.setdefault()`, and dictionary set views across transaction datasets.


In [17]:
# Solution:
tx_dict = {}
with open(csv_path, 'r') as f:
    f.readline()
    for _ in range(30):
        row = f.readline().strip().split(',')
        tx_dict[row[0]] = row[5]
        
override_dict = {row[0]: 'FORCE_CLEARED' for row in [row]}
merged_dict = tx_dict | override_dict
print('Sample merged ID:', list(merged_dict.keys())[0], 'Status:', merged_dict[list(merged_dict.keys())[0]])


Sample merged ID: TX110686 Status: Failed


### Q2: Fast Transaction Frequency Aggregation with `setdefault`
**Explanation**: **Scenario**: Aggregate transaction counts and total transaction volume by geographic region from the dataset using `.setdefault()` to build a high-performance analytics ledger.

**Syntax**: `ledger.setdefault(region, {'count': 0, 'volume': 0.0})`

In [18]:
# Solution:
region_frequencies_dict = {}
with open(csv_path, 'r') as f:
    f.readline()
    for _ in range(50):
        row = f.readline().strip().split(',')
        region = row[9].strip()
        region_frequencies_dict[region] = region_frequencies_dict.setdefault(region, 0) + 1
print('Region frequencies:', region_frequencies_dict)


Region frequencies: {'North': 11, 'East': 10, 'West': 10, 'South': 18, 'south': 1}
